# Lab 01: Introduction to Pandas & Data Types

## Instructions

<div class="alert alert-block alert-info"> <b>Grading</b> This will be graded as your Lab 1 grade. You should submit the entire notebook, exported as a pdf AND as an ipynb, to Canvas by Sunday at midnight.  </div>

Follow the prompts in this notebook.

Make use of code provided.

Respond with code to generate your answers.

If asked to provide a specific response, create a Markdown cell after your code cell(s) with the response.

This lab has four parts:

- **Part 1** gets you working with a real dataset in a Pandas `DataFrame`, and has you inventory its columns by data type and scale (nominal, ordinal, interval, or ratio) — the vocabulary from Lecture 4.
- **Part 2** has you visualize a nominal and an ordinal attribute.
- **Part 3** covers wide vs. long data tables — another idea from Lecture 4 — and how to reshape between them with `pd.melt()` and `.pivot()`.
- **Part 4** combines a nominal attribute with a quantitative one using `.groupby()`.

## Part 1: Loading data and inventorying its columns

## Task 1

Pass the `data_src` variable to Pandas' `.read_csv()` method to get a data frame of data about volcanoes.

Call the data frame `df`.

Show the first five rows of the data frame.

In [ ]:
import pandas as pd

data_src = "https://gist.githubusercontent.com/mattkram/9684863843254402942dfede27af2cb7/raw/2590dd8185b833aacf247c0595edbb07a025a6d7/Smithsonian_VOTW_Holocene_Volcanoes.csv"

## Task 2
Using a Raw cell below, create a comma separated list of column names, data types, and data scale.

You can get the columns names by accessing the `columns` attribute of the data frame, like so: `df.columns`.

For data types, infer these from visual inspection and Python's keywords, i.e. `str`, `int`, `float`, and `bool`.

For data scale, use the four categories from Lecture 4: `nominal`, `ordinal`, `interval` (Q-interval), or `ratio` (Q-ratio).

Treat any boolean data as nominal.

Treat dates and URLs as strings, and treat dates as interval (there's no true zero — "year 0" doesn't mean the absence of a date).

If the column data doesn't correspond to a scale, write `NA`.

Disregard the `GeoLocation` column.

## Task 3

How many ordinal columns are there?

## Task 4


How many ratio columns are there?

## Part 2: Visualizing by scale

Lecture 4 introduced four attribute scales — nominal, ordinal, Q-interval, and Q-ratio — and the idea that the right chart depends on which scale an attribute is measured on. We'll only touch on charting briefly here; Lab 2 goes much deeper into visualization.

### Nominal: `Region`

## Task 5

Using Pandas' `.value_counts()` method, how many regions are represented in the dataset and which one has the most observations associated with it?

Call the function by passing the `Region` column name to it.

## Task 6


Create a bar chart from results of the value count in the previous task using Pandas' plotting function `.plot.bar()`.

Be careful to select the column as a list and not a scalar value.

### Ordinal: `Evidence_Category`

Unlike a nominal attribute, an ordinal attribute *does* have a natural order — but Pandas doesn't know that order unless you tell it. `Evidence_Category` records how confident volcanologists are that a volcano erupted during the Holocene, from least to most certain:

`Evidence Uncertain` → `Evidence Credible` → `Eruption Dated` → `Eruption Observed`

(A fifth value, `Unrest / Holocene`, marks volcanoes with unrest but no confirmed eruption — treat it as coming before `Evidence Uncertain`.)

## Task 7


Run the cell below, which uses `pd.Categorical()` to tell Pandas the correct order for `Evidence_Category`, storing it in a new column `Evidence_Ordered`.

Then use `.value_counts()` **with `sort=False`**  on `Evidence_Ordered` to get the counts back out in the order you defined, rather than sorted by frequency. Finally, plot the result with `.plot.bar()`.

Compare the bar order here to what you'd get by calling `.value_counts()` on the original `Evidence_Category` column (no ordering) — why does the order matter for an ordinal attribute in a way it didn't for the nominal `Region` in Task 6?

In [ ]:
evidence_order = [
    "Unrest / Holocene",
    "Evidence Uncertain",
    "Evidence Credible",
    "Eruption Dated",
    "Eruption Observed",
]
df["Evidence_Ordered"] = pd.Categorical(df["Evidence_Category"], categories=evidence_order, ordered=True)

## Part 3: Wide vs. long data tables

Lecture 4 also distinguished **wide** and **long** data tables. In a long ("tidy") table, every variable is a column and every observation is a row — like `df` itself. In a wide table, values of one variable get spread out across *many* columns, the way the lecture's WHO tuberculosis example packed each sex/age group (`m_0-14`, `f_15-24`, ...) into its own column instead of a single `group` column.

The cell below builds a small **toy** table in that same style — not real WHO data, just illustrative counts — with one row per `country`/`year` and one column per demographic group.

In [ ]:
tb_wide = pd.DataFrame({
    "country": ["Afghanistan", "Afghanistan", "Brazil", "Brazil"],
    "year": [1999, 2000, 1999, 2000],
    "m_0-14": [5, 6, 37, 41],
    "m_15-24": [10, 12, 212, 246],
    "f_0-14": [3, 4, 21, 25],
    "f_15-24": [8, 9, 180, 190],
})
tb_wide

## Task 8


`tb_wide` is wide: the demographic group is smeared across four column *names* instead of living in its own column. Use `tb_wide.melt()`, passing `id_vars=["country", "year"]`, `var_name="group"`, and `value_name="cases"`, to reshape it to long format. Call the result `tb_long`.

## Task 9


Now reshape `tb_long` back to wide with `.pivot()`, using `index=["country", "year"]`, `columns="group"`, and `values="cases"`. (You'll want `.reset_index()` afterward to turn `country`/`year` back into columns instead of an index.)

Confirm your result has the same data as the original `tb_wide` (the column order may differ — that's fine).

## Task 10


Now apply the same idea to `df` itself. Use `pd.crosstab(df["Region"], df["Evidence_Category"])` to build a **wide** table: one row per region, one column per evidence category, with volcano counts in each cell. Call it `evidence_wide`.

Then melt `evidence_wide` to a **long** table with one row per region/category combination, called `evidence_long`. (Hint: `.reset_index()` first so `Region` becomes a column again, then `.melt(id_vars="Region", var_name="Evidence_Category", value_name="count")`.)

In a markdown cell, answer both:

1. Using `evidence_wide`, which region has the most `Eruption Observed` volcanoes?
2. Using `evidence_long`, how many region/category combinations have a count of zero?

Which question was easier to answer in the wide format, and which was easier in the long format? Why?

## Part 4: Combining scales with `.groupby()`

A common way to combine scales is to use a nominal (or ordinal) attribute to group a quantitative one — the same move you saw in Lecture 4's discussion of dimensions (usually nominal/ordinal) and measures (usually quantitative).

## Task 11


Use `df.groupby("Region")["Elevation"]` to split `Elevation` into one group per region. Build a dictionary mapping each region name to its elevation values (with the index reset, so lengths don't need to match), pass that dictionary to `pd.DataFrame()`, and plot the result with `.plot.box()`.

In a markdown cell, name one region that tends to have noticeably higher or lower elevations than the others, based on the plot.

## Wrap-up

(This is graded as a completion; any answer you give will be considered full points, and you can only lose points by not answering it.)
<div class="alert alert-block alert-warning"> Lecture 4 covered items/attributes/relationships, wide vs. long tables, and the four attribute scales. Which of these concepts do you think would be hardest to explain to a friend who's never taken this class, and why? </div>

